<table style="width: 100%; border-collapse: collapse; border: none; background: #f8fafc; border-left: 6px solid #1e3a8a; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #0f172a; font-size: 2.1em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        Taller Hands-On: Tu Primer Proyecto Completo de ML 🧪
      </h1>
      <p style="margin: 6px 0 0 0; color: #1e3a8a; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #64748b; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #1e3a8a; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        Taller Práctico Evaluativo · Módulo 00
      </span><br>
      <span style="color: #64748b; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #2563eb; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/homeworks/00_Introduccion_ML_Hands_On.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## Objetivos de Aprendizaje 🔎

Este taller consolida el **Módulo 00: Introducción al Machine Learning** ([README del módulo](../00%20-%20Introduccion%20al%20Machine%20Learning/README.md)) con un problema **nuevo**: decidir si un lote de leche recibido en un centro de acopio de Boyacá será **rechazado** o aceptado.

Al terminar serás capaz de:

1. **Formular** un problema de ML: tipo de problema, paradigma y variable objetivo.
2. **Detectar la fuga de datos** (*data leakage*) en una columna que "sabe demasiado" y excluirla.
3. **Dividir** los datos de forma estratificada y fijar una **línea base** contra la cual comparar.
4. **Construir** un `ColumnTransformer` + `Pipeline` que imputa, escala y codifica **ajustándose solo con el conjunto de entrenamiento**.
5. **Evaluar con honestidad:** validación cruzada sobre entrenamiento y una única medición final en prueba.

> 📚 **Repasa si lo necesitas:** [Introducción y Panorama del ML](../00%20-%20Introduccion%20al%20Machine%20Learning/00_Introduccion_y_Panorama_del_ML.ipynb) · [Tipos de Problemas](../00%20-%20Introduccion%20al%20Machine%20Learning/02_Tipos_de_Problemas_en_ML.ipynb) · [Importancia del Preprocesamiento](../00%20-%20Introduccion%20al%20Machine%20Learning/03_Importancia_del_Preprocesamiento.ipynb).

---
## Instrucciones 📋

1. **Resuelve de forma autónoma:** en cada reto completa la celda de código marcada con `# TODO:`. Este cuaderno **no trae soluciones**.
2. **Autoverificación:** debajo de cada reto hay una celda con `assert`. Si tu solución es correcta imprime ✅; si no, el mensaje de error te dice qué revisar. No la modifiques.
3. **Puntaje:** cada reto indica su puntaje sugerido; el total del taller suma **100 puntos**.
4. Antes de entregar usa *Restart & Run All*: el cuaderno debe correr de principio a fin, sin errores y **sin `# TODO` pendientes**.
5. Todo funciona **sin internet y sin archivos externos** (datos sintéticos con semilla fija). Bibliotecas: `numpy`, `pandas`, `scikit-learn`.

In [ ]:
# ============================================================
# Preparación (no modificar): imports y datos sintéticos
# ============================================================
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, accuracy_score

warnings.filterwarnings("ignore")


def generar_lotes(n=1500, seed=7):
    # Lotes de leche en un centro de acopio. Objetivo: rechazado (1) / aceptado (0).
    rng = np.random.default_rng(seed)
    temp = rng.normal(8, 3, n)                  # °C al recibir el lote
    acidez = rng.normal(16, 1.2, n)             # grados Dornic
    grasa = rng.normal(3.6, 0.35, n)            # % de grasa
    dist = rng.gamma(4, 6, n)                   # km desde la finca
    altitud = rng.normal(2800, 250, n)          # msnm de la finca
    ordeno = rng.choice(["manual", "mecanico"], n, p=[.45, .55])
    muni = rng.choice(["Tunja", "Samaca", "Ventaquemada", "Sora"], n)
    z = (0.9 * (temp - 8) / 3 + 1.0 * (acidez - 16) / 1.2 - 0.6 * (grasa - 3.6) / 0.35
         + 0.5 * (dist - 24) / 12 + 0.7 * (ordeno == "manual") + 0.3 * (muni == "Sora") - 1.6)
    y = rng.binomial(1, 1 / (1 + np.exp(-z)))
    # Prueba de antibióticos: se hace DESPUÉS de decidir el rechazo (coincide con él el 96 % de las veces)
    prueba = np.where(rng.random(n) < 0.04, 1 - y, y)
    df = pd.DataFrame({"temperatura_c": temp, "acidez_dornic": acidez, "grasa_pct": grasa,
                       "distancia_km": dist, "altitud_msnm": altitud, "tipo_ordeno": ordeno,
                       "municipio": muni, "prueba_antibiotico": prueba, "rechazado": y})
    for c in ["temperatura_c", "acidez_dornic", "grasa_pct"]:      # sensores que fallan
        df.loc[rng.random(n) < 0.08, c] = np.nan
    df.loc[rng.random(n) < 0.05, "tipo_ordeno"] = np.nan           # dato no registrado
    return df


df = generar_lotes()
print("Dimensiones:", df.shape, "| Proporción de rechazados:", round(df["rechazado"].mean(), 3))
display(df.head())
print(df.isna().sum()[df.isna().sum() > 0].to_string())

---
## Reto 1: Formular el problema, cazar la fuga y dividir 🎯

**Contexto:** el centro de acopio quiere predecir *antes de descargar el camión* si un lote será rechazado. Tú recibes la tabla `df` completa.

**Enunciado**
1. Indica el `tipo_problema` (`"clasificación"`, `"regresión"`, `"clustering"` o `"asociación"`) y el `paradigma` (`"supervisado"`, `"no supervisado"` o `"refuerzo"`).
2. Para cada columna numérica (excepto `rechazado`) calcula el **AUC** de usarla sola como puntaje para predecir `rechazado` (imputa los `NaN` con la mediana de la columna) y guárdalos en una `pd.Series` llamada `aucs`. Una columna con AUC (o 1 - AUC) cercano a 1 es sospechosa: guarda su nombre en `columna_fuga`.
3. Construye `X` (predictoras, **sin** el objetivo y **sin** la columna de fuga) e `y`.
4. Divide en `X_train, X_test, y_train, y_test`: **75 % / 25 %**, estratificado por `y`, `random_state=0`.

**Criterios de aceptación:** tipo y paradigma correctos; `aucs` con una entrada por columna numérica; la fuga identificada por los datos (no a ojo); partición estratificada con la proporción pedida.

**Puntaje sugerido:** 20 puntos.

In [ ]:
# TODO: completa cada línea (deja el nombre de las variables tal cual)
tipo_problema = None   # "clasificación" | "regresión" | "clustering" | "asociación"
paradigma = None       # "supervisado" | "no supervisado" | "refuerzo"

# Pista: recorre las columnas numéricas (sin 'rechazado'), rellena NaN con la mediana y usa roc_auc_score
aucs = None            # pd.Series: índice = nombre de columna, valor = AUC de esa columna sola
columna_fuga = None    # nombre (str) de la columna con información "del futuro"

X = None               # DataFrame de predictoras (sin objetivo ni fuga)
y = None               # Serie objetivo
X_train = X_test = y_train = y_test = None   # partición 75/25 estratificada, random_state=0

In [ ]:
# Autoverificación del Reto 1 (no modificar)
assert tipo_problema in ("clasificación", "clasificacion"), "El objetivo 'rechazado' es una etiqueta de dos clases: ¿qué tipo de problema es?"
assert paradigma == "supervisado", "Hay una columna objetivo con respuestas conocidas: ¿qué paradigma usa respuestas conocidas?"
assert isinstance(aucs, pd.Series) and len(aucs) == 6, "'aucs' debe ser una pd.Series con las 6 columnas numéricas predictoras (todas menos 'rechazado')."
assert columna_fuga == "prueba_antibiotico", "La columna de fuga es la que, SOLA, casi predice el objetivo (AUC cercano a 1): revisa 'aucs'."
assert X is not None and y is not None, "Define X e y."
assert columna_fuga not in X.columns and "rechazado" not in X.columns, "X no debe contener ni el objetivo ni la columna de fuga."
assert len(X_train) + len(X_test) == len(df) and abs(len(X_test) / len(df) - 0.25) < 0.01, "La partición debe ser 75 % entrenamiento / 25 % prueba."
assert abs(y_train.mean() - y_test.mean()) < 0.02, "Usa stratify=y para que la proporción de rechazados sea casi igual en ambas partes."
print("✅ Reto 1 superado: problema bien formulado, fuga excluida y partición estratificada.")

---
## Reto 2: Un preprocesador que no hace trampa 🧼

**Contexto:** los sensores fallan (hay `NaN`), las columnas tienen escalas muy distintas (`altitud_msnm` ~ 2800 frente a `grasa_pct` ~ 3.6) y hay dos variables de texto.

**Enunciado**
1. Guarda en `columnas_num` y `columnas_cat` las columnas numéricas y categóricas de `X_train` (listas de nombres).
2. Crea `preprocesador`, un `ColumnTransformer` con dos ramas:
   - **numéricas:** `SimpleImputer(strategy="median")` → `StandardScaler()`.
   - **categóricas:** `SimpleImputer(strategy="most_frequent")` → `OneHotEncoder(handle_unknown="ignore", sparse_output=False)`.
3. No lo ajustes aquí: la autoverificación lo ajusta **solo con `X_train`** y lo aplica a `X_test`.

**Criterios de aceptación:** sin `NaN` tras transformar; las numéricas quedan con media 0 y desviación 1 *en entrenamiento*; una columna por cada categoría (sin `drop`); mismo número de columnas en entrenamiento y prueba.

**Puntaje sugerido:** 25 puntos.

In [ ]:
columnas_num = None   # lista de nombres de columnas numéricas de X_train
columnas_cat = None   # lista de nombres de columnas categóricas de X_train

# TODO: arma las dos ramas (cada una un Pipeline de imputador + transformador) y únelas
preprocesador = None  # ColumnTransformer([("num", ..., columnas_num), ("cat", ..., columnas_cat)])

In [ ]:
# Autoverificación del Reto 2 (no modificar)
assert isinstance(columnas_num, list) and sorted(columnas_num) == sorted(X_train.select_dtypes("number").columns), "'columnas_num' debe listar las 5 columnas numéricas de X_train."
assert isinstance(columnas_cat, list) and sorted(columnas_cat) == ["municipio", "tipo_ordeno"], "'columnas_cat' debe listar 'tipo_ordeno' y 'municipio'."
assert isinstance(preprocesador, ColumnTransformer), "'preprocesador' debe ser un ColumnTransformer."
_denso = lambda M: np.asarray(M.toarray() if hasattr(M, "toarray") else M, dtype=float)
Zt = _denso(preprocesador.fit_transform(X_train))      # se ajusta SOLO con entrenamiento
Zp = _denso(preprocesador.transform(X_test))
assert not np.isnan(Zt).any() and not np.isnan(Zp).any(), "Quedaron NaN: falta imputar alguna rama."
n_esperado = len(columnas_num) + sum(X_train[c].nunique() for c in columnas_cat)
assert Zt.shape[1] == n_esperado == Zp.shape[1], f"Se esperaban {n_esperado} columnas (5 numéricas + una por categoría); obtuviste {Zt.shape[1]}."
estandarizadas = np.sum((np.abs(Zt.mean(axis=0)) < 1e-6) & (np.abs(Zt.std(axis=0) - 1) < 1e-6))
assert estandarizadas >= len(columnas_num), "Las columnas numéricas deben quedar con media 0 y desviación 1 en entrenamiento (¿falta el StandardScaler?)."
print("✅ Reto 2 superado: preprocesador ajustado solo con entrenamiento,", Zt.shape[1], "columnas.")

---
## Reto 3: Medir cuánto vale el preprocesamiento ⚖️

**Contexto:** el algoritmo $k$-NN decide por **distancias**; si una variable tiene una escala enorme, domina a todas las demás. Vas a verificarlo con números.

**Enunciado**
1. Crea `pipe`: `Pipeline` con el `preprocesador` del Reto 2 y un `KNeighborsClassifier(n_neighbors=15)`; ajústalo con `X_train, y_train`.
2. Crea `pipe_crudo`: `Pipeline` con **solo** `SimpleImputer(strategy="median")` y el mismo $k$-NN, usando **únicamente las columnas numéricas** (`X_train[columnas_num]`), sin escalar ni codificar; ajústalo.
3. Calcula `auc_pipe` y `auc_crudo`: el AUC en el conjunto de **prueba** de cada uno.

**Criterios de aceptación:** ambos pipelines ajustados; los AUC coinciden con una recomputación independiente; el pipeline completo supera al crudo por más de 0.05 de AUC.

**Puntaje sugerido:** 25 puntos.

In [ ]:
# TODO: arma y ajusta los dos pipelines, y calcula sus AUC en prueba
pipe = None         # Pipeline([("prep", preprocesador), ("knn", KNeighborsClassifier(n_neighbors=15))])
pipe_crudo = None   # Pipeline([("imp", SimpleImputer(strategy="median")), ("knn", KNeighborsClassifier(n_neighbors=15))])

auc_pipe = None     # AUC de `pipe` en X_test
auc_crudo = None    # AUC de `pipe_crudo` en X_test[columnas_num]

In [ ]:
# Autoverificación del Reto 3 (no modificar)
assert isinstance(pipe, Pipeline) and isinstance(pipe.steps[-1][1], KNeighborsClassifier), "'pipe' debe ser un Pipeline que termina en KNeighborsClassifier."
assert isinstance(pipe_crudo, Pipeline) and isinstance(pipe_crudo.steps[-1][1], KNeighborsClassifier), "'pipe_crudo' debe ser un Pipeline que termina en KNeighborsClassifier."
assert auc_pipe is not None and auc_crudo is not None, "Calcula auc_pipe y auc_crudo."
assert abs(auc_pipe - roc_auc_score(y_test, pipe.predict_proba(X_test)[:, 1])) < 1e-9, "auc_pipe no coincide con el AUC de 'pipe' en X_test (¿lo ajustaste con fit?)."
assert abs(auc_crudo - roc_auc_score(y_test, pipe_crudo.predict_proba(X_test[columnas_num])[:, 1])) < 1e-9, "auc_crudo no coincide con el AUC de 'pipe_crudo' en X_test[columnas_num]."
assert auc_pipe > auc_crudo + 0.05, "El pipeline completo debería superar al crudo por más de 0.05 de AUC: revisa que 'pipe' use el preprocesador con escalado."
print(f"✅ Reto 3 superado: AUC con preprocesamiento = {auc_pipe:.3f} vs. crudo = {auc_crudo:.3f}.")

---
## Reto 4: Evaluar con honestidad 🧾

**Contexto:** un buen reporte separa dos preguntas: *¿qué tan bien creo que funcionará?* (validación cruzada **solo con entrenamiento**) y *¿cómo funcionó de verdad?* (una única medición en prueba). Además, toda cifra necesita una **línea base**.

**Enunciado**
1. `cv_auc`: arreglo con los 5 AUC de `cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc")`, usando el `cv` ya definido (los datos de prueba **no** se tocan).
2. `auc_test`: AUC de `pipe` en el conjunto de prueba.
3. `acc_base`: exactitud en prueba de un `DummyClassifier(strategy="most_frequent")` ajustado con entrenamiento; `acc_pipe`: exactitud en prueba de `pipe`.

**Criterios de aceptación:** `cv_auc` reproducible con el `cv` dado; la estimación por validación cruzada y la medición en prueba difieren menos de 0.10; el pipeline supera la línea base de exactitud.

**Puntaje sugerido:** 15 puntos.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)   # no lo cambies

# TODO: completa
cv_auc = None     # np.ndarray con 5 AUC (validación cruzada sobre X_train, y_train)
auc_test = None   # AUC de `pipe` en prueba
acc_base = None   # exactitud en prueba del DummyClassifier("most_frequent")
acc_pipe = None   # exactitud en prueba de `pipe`

In [ ]:
# Autoverificación del Reto 4 (no modificar)
assert cv_auc is not None and len(cv_auc) == 5, "'cv_auc' debe tener 5 valores (uno por partición)."
_ref = cross_val_score(pipe, X_train, y_train, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=0), scoring="roc_auc")
assert np.allclose(cv_auc, _ref), "cv_auc no coincide con cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc'): ¿usaste solo el conjunto de entrenamiento y el 'cv' dado?"
assert auc_test is not None and abs(cv_auc.mean() - auc_test) < 0.10, "La estimación por validación cruzada y el AUC en prueba deberían ser parecidos (< 0.10 de diferencia)."
assert acc_base is not None and abs(acc_base - max(y_test.mean(), 1 - y_test.mean())) < 0.02, "La línea base 'most_frequent' acierta ≈ la proporción de la clase mayoritaria."
assert acc_pipe is not None and acc_pipe > acc_base, "El modelo debería superar a la línea base de exactitud."
print(f"✅ Reto 4 superado: CV = {cv_auc.mean():.3f} ± {cv_auc.std():.3f} | prueba = {auc_test:.3f} | exactitud {acc_pipe:.3f} vs. base {acc_base:.3f}.")

---
## Reto 5: Conclusiones 📝

**Puntaje sugerido:** 15 puntos (sin código). Escribe **5 a 8 líneas** en la celda de abajo interpretando *tus* resultados. Debe responder:

- ¿Qué tipo de problema y paradigma es este, y por qué?
- ¿Por qué `prueba_antibiotico` no se puede usar para el objetivo del centro de acopio, aunque "mejoraría" el modelo?
- ¿Qué diferencia de AUC obtuviste entre `pipe` y `pipe_crudo`, y qué variable crees que dominaba las distancias en el modelo crudo?
- ¿La estimación por validación cruzada se parece al resultado en prueba? ¿Qué te dice la comparación con la línea base de exactitud?

---
#### ✍️ Tus conclusiones

_Escribe aquí tus 5-8 líneas. Reemplaza este texto por tu análisis._

---
### 🧠 Rúbrica de evaluación

| Criterio | Puntos |
|---|---|
| Reto 1: formulación, detección de la fuga y partición estratificada | 20 |
| Reto 2: `ColumnTransformer` correcto (imputar, escalar, codificar) ajustado solo con entrenamiento | 25 |
| Reto 3: pipelines comparados con AUC en prueba y recomputación coherente | 25 |
| Reto 4: validación cruzada reproducible, prueba y línea base | 15 |
| Reto 5: conclusiones argumentadas con tus cifras (5-8 líneas) | 15 |
| **Total** | **100** |

### ✅ Checklist de entrega

- [ ] Reinicié el kernel y ejecuté *Run All* sin errores.
- [ ] No quedan `# TODO` ni variables en `None`.
- [ ] Las cinco celdas de autoverificación imprimen ✅.
- [ ] El preprocesamiento vive dentro del `Pipeline` (nada se ajustó con datos de prueba).
- [ ] Escribí mis conclusiones con cifras de mis propios resultados.

---
<div align="center">
  <p style="font-size: 0.9em; color: #64748b;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Asignatura: Machine Learning</i>
  </p>
</div>